
# Computational Spintronics — Notebook 01
## LLG Magnetization Dynamics → MTJ Resistance → TMR

A self-contained macrospin introduction connecting magnetic dynamics to an electrical MTJ observable. This is a single-domain model, not a spatial micromagnetic simulation.

**Learning goals**
- Implement the Gilbert equation in explicit Landau–Lifshitz form.
- Integrate magnetization dynamics and visualize precession and damping.
- Check unit-vector preservation, torque orthogonality, and energy relaxation.
- Map the free-layer direction to MTJ resistance and TMR.



## 1. Model and equations

The normalized magnetization is \(\mathbf m=\mathbf M/M_s\), with \(|\mathbf m|=1\).
The Gilbert equation is

\[
\frac{d\mathbf m}{dt}=-\gamma\mu_0\mathbf m\times\mathbf H_{\rm eff}
+\alpha\mathbf m\times\frac{d\mathbf m}{dt}.
\]

Solving for the derivative gives

\[
\frac{d\mathbf m}{dt}=-\frac{\gamma\mu_0}{1+\alpha^2}
\left[\mathbf m\times\mathbf H_{\rm eff}
+\alpha\mathbf m\times(\mathbf m\times\mathbf H_{\rm eff})\right].
\]

We use a uniaxial effective field
\[
\mathbf H_{\rm eff}=H_km_z\hat{\mathbf z}+\mathbf H_{\rm ext}.
\]


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(precision=5, suppress=True)
plt.rcParams["figure.figsize"] = (8, 4.8)

mu0 = 4*np.pi*1e-7       # H/m
gamma = 1.760859e11      # rad/(s*T)
alpha = 0.02
Ms = 1.0e6               # A/m
Hk = 8.0e5               # A/m
Hext = np.array([0.0, 0.0, 0.0])
ez = np.array([0.0, 0.0, 1.0])

print(f"alpha={alpha}, Ms={Ms:.2e} A/m, Hk={Hk:.2e} A/m")


## 2. Initial state and LLG implementation

In [ ]:

theta0, phi0 = np.deg2rad([25.0, 20.0])
m0 = np.array([np.sin(theta0)*np.cos(phi0),
               np.sin(theta0)*np.sin(phi0),
               np.cos(theta0)])

def effective_field(m):
    return Hk*m[2]*ez + Hext

def llg_rhs(t, m):
    H = effective_field(m)
    mxH = np.cross(m, H)
    return -gamma*mu0/(1+alpha**2) * (
        mxH + alpha*np.cross(m, mxH)
    )

print("m0 =", m0, "|m0| =", np.linalg.norm(m0))
print("m · dm/dt =", np.dot(m0, llg_rhs(0, m0)))


## 3. Integrate the dynamics

In [ ]:

t_end = 2e-9
t_eval = np.linspace(0, t_end, 4000)
sol = solve_ivp(llg_rhs, (0, t_end), m0, t_eval=t_eval,
                method="DOP853", rtol=1e-9, atol=1e-11)
if not sol.success:
    raise RuntimeError(sol.message)
t, m = sol.t, sol.y.T
print(sol.message, "\nFinal m =", m[-1])


In [ ]:

fig, ax = plt.subplots()
for j, label in enumerate([r"$m_x$", r"$m_y$", r"$m_z$"]):
    ax.plot(t*1e9, m[:, j], label=label)
ax.set(xlabel="Time (ns)", ylabel="Normalized magnetization",
       title="LLG magnetization dynamics")
ax.grid(alpha=.3); ax.legend(); plt.show()


## 4. Unit-sphere trajectory

In [ ]:

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
u = np.linspace(0, 2*np.pi, 70)
v = np.linspace(0, np.pi, 36)
ax.plot_surface(np.outer(np.cos(u), np.sin(v)),
                np.outer(np.sin(u), np.sin(v)),
                np.outer(np.ones_like(u), np.cos(v)),
                alpha=.10, linewidth=0)
ax.plot(m[:,0], m[:,1], m[:,2], lw=2)
ax.scatter(*m[0], label="initial")
ax.scatter(*m[-1], label="final")
ax.set(xlabel="$m_x$", ylabel="$m_y$", zlabel="$m_z$",
       title="Macrospin trajectory")
ax.legend(); plt.show()



## 5. Energy and numerical checks

For this model, the energy density is

\[
E/V=-\frac{\mu_0M_sH_k}{2}m_z^2-\mu_0M_s\mathbf H_{\rm ext}\cdot\mathbf m.
\]


In [ ]:

def energy_density(mm):
    return (-0.5*mu0*Ms*Hk*mm[...,2]**2
            -mu0*Ms*np.sum(mm*Hext, axis=-1))

E = energy_density(m)
fig, ax = plt.subplots()
ax.plot(t*1e9, E)
ax.set(xlabel="Time (ns)", ylabel="Energy density (J/m³)",
       title="Energy relaxation")
ax.grid(alpha=.3); plt.show()

norm_error = np.max(np.abs(np.linalg.norm(m, axis=1)-1))
dm = np.array([llg_rhs(tt, mm) for tt, mm in zip(t, m)])
perp_error = np.max(np.abs(np.sum(m*dm, axis=1)))
energy_increase = np.max(np.diff(E))
print(f"Maximum norm error: {norm_error:.3e}")
print(f"Maximum |m·dm/dt|: {perp_error:.3e}")
print(f"Maximum energy step increase: {energy_increase:.3e} J/m³")
assert norm_error < 1e-6
assert perp_error < 1e-4
assert energy_increase < 1e-4
assert m[-1,2] > m0[2]
print("Validation passed.")



## 6. Easy-axis equilibrium states

With zero external field, \(+\hat z\) and \(-\hat z\) are equilibrium states.


In [ ]:

m_up = np.array([0., 0., 1.])
m_down = np.array([0., 0., -1.])
print("Torque at +z:", llg_rhs(0, m_up))
print("Torque at -z:", llg_rhs(0, m_down))
assert np.linalg.norm(llg_rhs(0, m_up)) < 1e-12
assert np.linalg.norm(llg_rhs(0, m_down)) < 1e-12



## 7. MTJ resistance and TMR

Let the reference layer point along \(\mathbf p=\hat z\), so
\(\cos\theta=\mathbf m\cdot\mathbf p\). A compact angular conductance model is

\[
G(\theta)=G_P\frac{1+\eta\cos\theta}{1+\eta},\qquad
\eta=\frac{\mathrm{TMR}}{2+\mathrm{TMR}},\quad G_P=1/R_P.
\]

It yields \(R(0)=R_P\) and \(R(\pi)=R_P(1+\mathrm{TMR})=R_{AP}\).


In [ ]:

p_ref = np.array([0., 0., 1.])
RP = 5000.0
TMR = 1.50
eta = TMR/(2+TMR)
GP = 1/RP

def mtj_resistance(mm):
    mm = np.asarray(mm)
    c = np.clip(np.sum(mm*p_ref, axis=-1), -1, 1)
    return 1/(GP*(1+eta*c)/(1+eta))

R = mtj_resistance(m)
R_P = mtj_resistance(m_up)
R_AP = mtj_resistance(m_down)
print(f"R_P={R_P:.1f} Ω; R_AP={R_AP:.1f} Ω; measured TMR={R_AP/R_P-1:.3f}")
assert np.isclose(R_P, RP)
assert np.isclose(R_AP, RP*(1+TMR))


In [ ]:

fig, ax1 = plt.subplots()
ax1.plot(t*1e9, m[:,2], label=r"$m_z$")
ax1.set(xlabel="Time (ns)", ylabel=r"$m_z$",
        title="Magnetic state mapped to MTJ resistance")
ax1.grid(alpha=.3)
ax2 = ax1.twinx()
ax2.plot(t*1e9, R/1e3, "--", label="MTJ resistance")
ax2.set_ylabel("Resistance (kΩ)")
plt.show()


## 8. Static angular resistance

In [ ]:

theta = np.linspace(0, np.pi, 500)
m_theta = np.column_stack([np.sin(theta), np.zeros_like(theta), np.cos(theta)])
R_theta = mtj_resistance(m_theta)
fig, ax = plt.subplots()
ax.plot(np.rad2deg(theta), R_theta/1e3)
ax.axhline(R_P/1e3, ls="--", label="P")
ax.axhline(R_AP/1e3, ls="--", label="AP")
ax.set(xlabel="Relative angle θ (degrees)", ylabel="Resistance (kΩ)",
       title="Angular MTJ resistance")
ax.grid(alpha=.3); ax.legend(); plt.show()



## 9. Takeaways and next notebooks

The core chain is

\[
\boxed{\mathbf H_{\rm eff}\rightarrow\mathbf m(t)
\rightarrow\cos\theta(t)\rightarrow R_{\rm MTJ}(t)}
\]

This notebook establishes a deterministic macrospin baseline. It does **not** yet include thermal noise, spin-transfer torque, read/write circuitry, or spatial nonuniformity.

Suggested progression:
- **02:** thermal stochastic LLG
- **03:** Slonczewski spin-transfer torque
- **04:** STT-MTJ switching and switching statistics
- **05:** Verilog-A compact MTJ model
- **06:** 1T–1MTJ cell and SPICE read/write simulations
